# Cross-validation, part 2: the CNN, with and without the HMM

The same 5 folds of people as `cv_trees.ipynb`, and the same CNN design as Stage 2. For each fold:

1. Training people = the other 4 folds (about 121 people). 20% of them are set aside for early stopping, as in Stage 2
2. Train the CNN from scratch, on the **RTX 3060**
3. Learn the HMM from the training people's true labels, then predict and smooth the test fold

**Why Keras now runs on PyTorch:** TensorFlow on Windows cannot use the GPU, and the rule is no training on the CPU. Keras 3 can use PyTorch underneath instead, with the same layers and the same `.keras` files. Section 2 checks this by loading the Stage 2 model and getting the same answers.

**If it gets interrupted:** each finished fold is saved, and running the notebook again skips folds that are done.

**Kernel:** `.venv-tf` in this folder (now also with PyTorch). Run `cv_trees.ipynb` first.

## How this works: cross-validating the CNN, explained from scratch

The CNN itself (filters, layers, training, early stopping) is explained in Stage 2. This notebook re-tests the same design more thoroughly.

### Cross-validation, step by step

Stages 1 to 3 tested on **one** group of 30 people. With so few, a few unusual people can push a score up or down by luck. Cross-validation tests on **everyone**:

```
151 people, shuffled once (fixed seed), dealt into 5 folds of about 30

round 1:  [TEST] [train] [train] [train] [train]
round 2:  [train] [TEST] [train] [train] [train]
round 3:  [train] [train] [TEST] [train] [train]
round 4:  [train] [train] [train] [TEST] [train]
round 5:  [train] [train] [train] [train] [TEST]
```

In every round, a **brand-new model** is trained on 4 folds and tested on the 5th. After 5 rounds, every person has been tested exactly once, always by a model that never saw them.

What we get out:

| Result | What it tells us |
|---|---|
| **Pooled score** over all 151 people | A steadier estimate than the 30-person test |
| **Score in each fold** | How much the score moves just from *which* people are tested |
| **Score for each person** | Lets us compare two models person by person, on 151 people instead of 30 |

### Rules that keep it fair

| Rule | Why |
|---|---|
| Folds are made of **people**, not windows | Same reason as the test split: one person's windows look alike |
| All models use **the same folds** (`cv_folds()` in `common.py`) | Each model faces exactly the same test people |
| The HMM is learned again in each round, from that round's **training** people only | The HMM must not learn from the people it is tested on either |
| Settings are **not** tuned again | They came from earlier stages. Re-tuning inside every round would cost hours of GPU time. (Those settings were chosen on 121 of the 151 people, a small advantage every model shares) |
| Training on the **GPU only** | The no-CPU-training rule |

### What's different for the CNN

1. **A new CNN is trained from scratch in every round.** Same design, same settings, same starting seed
2. **Early stopping needs its own people.** In each round, 24 of the training people are set aside as validation (they decide when to stop). The CNN learns from the other ~97. The test fold is never used for this
3. **Keras now runs on PyTorch.** TensorFlow can't use the GPU on Windows. Keras 3 can switch its engine to PyTorch, which can. The layers and the saved `.keras` file are the same, and the notebook checks this by loading the Stage 2 model and getting the same predictions
4. **Each finished round is saved.** Training takes about an hour, so if it's interrupted, running again skips the rounds already done


In [1]:
import os
os.environ['KERAS_BACKEND'] = 'torch'              # WHY: must be set before Keras is imported
import time

import keras
import numpy as np
import pandas as pd
import torch
from sklearn.utils.class_weight import compute_class_weight

from common import DATA_DIR, N_FOLDS, RESULTS_DIR, SEED, cv_folds, evaluate, learn_hmm, smooth, split_people

# WHY: the rule is GPU only. Stop here instead of quietly training on the CPU.
assert keras.config.backend() == 'torch' and torch.cuda.is_available(), 'no GPU: stopping'
# WHY full 32-bit maths: by default PyTorch uses a faster, rougher format (TF32) on this GPU, which moved the
# Stage 2 model's probabilities by up to 0.005. Full 32-bit matches TensorFlow to about 0.000002, like Stage 2.
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
print('Keras', keras.__version__, '| PyTorch', torch.__version__, '|', torch.cuda.get_device_name(0))

Keras 3.15.1 | PyTorch 2.11.0+cu128 | NVIDIA GeForce RTX 3060 Laptop GPU


## 1. The windows on disk

The 121 Stage 2 training people are already in `processed/train_X.npy`. The other 30 go into `processed/test_X.npy` the same way, so all 151 can be read a block at a time without loading them into RAM.

In [2]:
folds = cv_folds()
everyone = sorted(folds)
train_ids, test_ids = split_people()

def load_labels(ids):
    """Labels, person id and start time. Fast, because the big X array is not read."""
    y, pid, start = [], [], []
    for p in ids:
        with np.load(DATA_DIR / f'{p}.npz') as f:
            y.append(f['y4'].astype(int)); start.append(f['start'])
        pid.append(np.full(len(y[-1]), p))
    return np.concatenate(y), np.concatenate(pid), np.concatenate(start)

TEST_FILE = DATA_DIR / 'test_X.npy'
if not TEST_FILE.exists():
    y_t, pid_t, _ = load_labels(test_ids)
    tmp = DATA_DIR / 'tmp_test_X.npy'              # WHY: a stopped copy can't look finished
    big = np.lib.format.open_memmap(tmp, mode='w+', dtype='float32', shape=(len(y_t), 1500, 3))
    for p in test_ids:
        with np.load(DATA_DIR / f'{p}.npz') as f:
            big[pid_t == p] = f['X']
    big.flush(); del big
    tmp.replace(TEST_FILE)

# one table of every window: which file it is in, and where
files = [DATA_DIR / 'train_X.npy', TEST_FILE]
stores = [np.load(f, mmap_mode='r') for f in files]
parts = [load_labels(train_ids), load_labels(test_ids)]
y_store = np.concatenate([p[0] for p in parts])
pid_store = np.concatenate([p[1] for p in parts])
file_of = np.concatenate([np.full(len(p[0]), i) for i, p in enumerate(parts)])
row_in_file = np.concatenate([np.arange(len(p[0])) for p in parts])
for s, p in zip(stores, parts):
    assert s.shape == (len(p[0]), 1500, 3)
for p in [train_ids[5], test_ids[0], test_ids[-1]]:  # the files match the originals
    m = np.flatnonzero(pid_store == p)
    with np.load(DATA_DIR / f'{p}.npz') as f:
        assert np.array_equal(stores[file_of[m[0]]][row_in_file[m]], f['X'])
print(f'{len(y_store):,} windows from {len(np.unique(pid_store))} people')

301,164 windows from 151 people


**How batches are read** (the same idea as Stage 2): each person's windows are cut into blocks of 64 neighbours, read with one disk read each. Each epoch the blocks are shuffled and poured into pools of 128 blocks; each pool is shuffled and handed out in batches of 256. Only one or two pools sit in RAM.

In [3]:
BATCH, BLOCK, POOL_BLOCKS = 256, 64, 128
WINDOW_BYTES = 1500 * 3 * 4
one_hot = np.eye(4, dtype='float32')

def read_block(rows):
    """Read neighbouring windows (all from one person, so one file) with one single disk read."""
    f = file_of[rows[0]]
    flat = np.fromfile(files[f], dtype='float32', count=len(rows) * 4500,
                       offset=stores[f].offset + int(row_in_file[rows[0]]) * WINDOW_BYTES)
    return flat.reshape(len(rows), 1500, 3)

def cut_blocks(idx):
    """Split row numbers into blocks of neighbours, never crossing from one person to another."""
    blocks = []
    for p in np.unique(pid_store[idx]):
        rows = idx[pid_store[idx] == p]
        assert (np.diff(rows) == 1).all()
        blocks += [rows[i:i + BLOCK] for i in range(0, len(rows), BLOCK)]
    return blocks

class Batches(keras.utils.PyDataset):
    """Hands Keras one batch at a time. Batch i = rows i*256 to (i+1)*256 of this epoch's shuffled order."""
    def __init__(self, idx, shuffle):
        super().__init__()
        self.blocks, self.shuffle, self.n, self.epoch = cut_blocks(idx), shuffle, len(idx), 0
        self.plan()

    def plan(self):
        # WHY shuffle every epoch: neighbouring windows are nearly identical, a batch should mix people and classes
        rng = np.random.default_rng([SEED, self.epoch])
        order = rng.permutation(len(self.blocks)) if self.shuffle else np.arange(len(self.blocks))
        self.pools = [[self.blocks[j] for j in order[i:i + POOL_BLOCKS]] for i in range(0, len(order), POOL_BLOCKS)]
        self.ends = np.cumsum([sum(len(b) for b in pool) for pool in self.pools])   # where each pool ends
        self.cache = {}

    def load_pool(self, k):
        if k not in self.cache:
            if len(self.cache) >= 2:                   # keep at most 2 pools in RAM
                self.cache.pop(min(self.cache))
            X = np.concatenate([read_block(b) for b in self.pools[k]])
            rows = np.concatenate(self.pools[k])
            if self.shuffle:
                mix = np.random.default_rng([SEED, self.epoch, k]).permutation(len(rows))
                X, rows = X[mix], rows[mix]
            self.cache[k] = (X, rows)
        return self.cache[k]

    def __len__(self):
        return int(np.ceil(self.n / BATCH))

    def __getitem__(self, i):
        a, b = i * BATCH, min((i + 1) * BATCH, self.n)
        Xs, rows = [], []
        for k in range(np.searchsorted(self.ends, a, side='right'), np.searchsorted(self.ends, b - 1, side='right') + 1):
            X, r = self.load_pool(k)
            first = self.ends[k - 1] if k else 0
            lo, hi = max(a, first) - first, min(b, self.ends[k]) - first
            Xs.append(X[lo:hi]); rows.append(r[lo:hi])
        rows = np.concatenate(rows)
        return np.concatenate(Xs), one_hot[y_store[rows]]

    def on_epoch_end(self):
        if self.shuffle:
            self.epoch += 1
            self.plan()

# check: every window comes out exactly once per epoch, with its own label, in two different epochs
check_idx = np.flatnonzero(np.isin(pid_store, everyone[:6]))
data = Batches(check_idx, shuffle=True)
lookup = {stores[file_of[r]][row_in_file[r], :20].tobytes(): y_store[r] for r in check_idx}
for epoch in range(2):
    seen = 0
    for i in range(len(data)):
        xb, yb = data[i]
        for x, label in zip(xb, yb.argmax(axis=1)):
            assert lookup[x[:20].tobytes()] == label
        seen += len(xb)
    assert seen == len(check_idx)
    first_batch = data[0][0][:3].copy() if epoch == 0 else first_batch
    data.on_epoch_end()
assert not np.array_equal(data[0][0][:3], first_batch)   # a new order each epoch
print('batch checks passed')

batch checks passed


## 2. The CNN: the Stage 2 design, now on the GPU

First a check that PyTorch-Keras computes the same thing as TensorFlow-Keras: load the Stage 2 model and predict one test person. The answers must match the saved Stage 2 predictions.

In [4]:
def conv_block(x, filters, width):
    x = keras.layers.Conv1D(filters, width, padding='same', use_bias=False)(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.ReLU()(x)
    return keras.layers.MaxPooling1D(4)(x)

def make_cnn():
    inputs = keras.Input((1500, 3))
    x = conv_block(inputs, 32, 9)
    x = conv_block(x, 64, 7)
    x = conv_block(x, 64, 5)
    x = keras.layers.GlobalAveragePooling1D()(x)
    outputs = keras.layers.Dense(4, activation='softmax')(x)
    return keras.Model(inputs, outputs, name='silverguard_cnn')

old = keras.models.load_model(RESULTS_DIR / 'cnn_4class.keras')
stage2 = np.load(RESULTS_DIR / 'stage2_predictions.npz')
with np.load(DATA_DIR / f'{test_ids[0]}.npz') as f:
    gap = np.abs(old.predict(f['X'], verbose=0) - stage2['probs'][stage2['pid'] == test_ids[0]]).max()
assert gap < 1e-4, gap
print(f'Stage 2 model under PyTorch: largest difference from TensorFlow {gap:.1e}')

cnn = make_cnn()
assert cnn.count_params() == old.count_params() == 36_580
assert cnn.weights[0].value.device.type == 'cuda', 'model is not on the GPU'
print('model on the GPU: yes,', cnn.count_params(), 'parameters')

Stage 2 model under PyTorch: largest difference from TensorFlow 1.1e-06
model on the GPU: yes, 36580 parameters


## 3. Cross-validation

Each fold trains a new CNN, up to 40 epochs, stopping when the validation people's macro-F1 has not improved for 6 epochs (as in Stage 2). About 10 to 20 minutes per fold on the GPU.

In [5]:
y_all, pid_all, start_all = load_labels(everyone)     # the order used by cv_trees: by person, then time
fold_all = pd.Series(pid_all).map(folds).to_numpy()
trees = np.load(RESULTS_DIR / 'cv_trees_predictions.npz')
assert (trees['pid'] == pid_all).all() and (trees['start'] == start_all).all() and (trees['fold'] == fold_all).all()

fold_store = pd.Series(pid_store).map(folds).to_numpy()
t_all = time.time()
for k in range(N_FOLDS):
    out_file = RESULTS_DIR / f'cv_cnn_fold{k}.npz'
    if out_file.exists():
        print(f'fold {k}: already done'); continue

    train_people = sorted(p for p in everyone if folds[p] != k)
    val_people = sorted(str(p) for p in np.random.default_rng(SEED + k).choice(train_people, size=24, replace=False))
    fit_idx = np.flatnonzero((fold_store != k) & ~np.isin(pid_store, val_people))
    val_idx = np.flatnonzero(np.isin(pid_store, val_people))
    assert not set(pid_store[fit_idx]) & set(val_people) and not set(val_people) & {p for p in everyone if folds[p] == k}

    keras.utils.set_random_seed(SEED)              # WHY: same starting weights and batch order every fold
    cnn = make_cnn()
    weights = compute_class_weight('balanced', classes=np.arange(4), y=y_store[fit_idx])
    cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss='categorical_crossentropy',
                metrics=['accuracy', keras.metrics.F1Score(average='macro', name='macro_f1')])
    t0 = time.time()
    history = cnn.fit(Batches(fit_idx, shuffle=True), validation_data=Batches(val_idx, shuffle=False),
                      epochs=40, class_weight={i: float(w) for i, w in enumerate(weights)}, verbose=2,
                      # WHY shuffle=False: Batches already shuffles every epoch. Keras' own shuffle jumbles the
                      # batch order, so each batch would need a different pool and re-read it from disk (12x slower)
                      shuffle=False,
                      callbacks=[keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=6,
                                                               restore_best_weights=True),
                                 keras.callbacks.CSVLogger(str(RESULTS_DIR / f'cv_cnn_fold{k}_history.csv'))])
    log = pd.DataFrame(history.history)
    best = int(log['val_macro_f1'].idxmax())

    # predict the test fold one person at a time, and smooth with this fold's HMM
    te = fold_all == k
    probs = np.zeros((te.sum(), 4))
    for p in np.unique(pid_all[te]):
        with np.load(DATA_DIR / f'{p}.npz') as f:
            probs[pid_all[te] == p] = cnn.predict(f['X'], batch_size=BATCH, verbose=0)
    start_p, transition = learn_hmm(y_all[~te], pid_all[~te])
    cnn.save(RESULTS_DIR / f'cv_cnn_fold{k}.keras')
    np.savez(out_file, probs=probs, pred_hmm=smooth(probs, pid_all[te], start_p, transition))
    print(f'fold {k}: {(time.time() - t0) / 60:.1f} min, {len(log)} epochs, best epoch {best + 1} '
          f'(val macro-F1 {log.val_macro_f1[best]:.3f}); test kappa {evaluate("", y_all[te], probs.argmax(1))[0]["kappa"]:.3f}')
print(f'total {(time.time() - t_all) / 60:.1f} min')

Epoch 1/40


754/754 - 28s - 38ms/step - accuracy: 0.7011 - loss: 0.7249 - macro_f1: 0.6586 - val_accuracy: 0.6518 - val_loss: 0.8597 - val_macro_f1: 0.6020


Epoch 2/40


754/754 - 29s - 39ms/step - accuracy: 0.7408 - loss: 0.6565 - macro_f1: 0.6942 - val_accuracy: 0.7057 - val_loss: 0.7329 - val_macro_f1: 0.6537


Epoch 3/40


754/754 - 30s - 40ms/step - accuracy: 0.7597 - loss: 0.6229 - macro_f1: 0.7132 - val_accuracy: 0.7653 - val_loss: 0.6438 - val_macro_f1: 0.7146


... (195 outputs left out)

765/765 - 31s - 40ms/step - accuracy: 0.8268 - loss: 0.4800 - macro_f1: 0.7837 - val_accuracy: 0.7847 - val_loss: 0.5734 - val_macro_f1: 0.7132


Epoch 22/40


765/765 - 30s - 40ms/step - accuracy: 0.8274 - loss: 0.4773 - macro_f1: 0.7860 - val_accuracy: 0.7351 - val_loss: 0.7410 - val_macro_f1: 0.6791


fold 4: 11.6 min, 22 epochs, best epoch 16 (val macro-F1 0.723); test kappa 0.718
total 52.0 min


## 4. Put the folds together

In [6]:
probs_cnn, pred_hmm = np.zeros((len(y_all), 4)), np.full(len(y_all), -1)
for k in range(N_FOLDS):
    with np.load(RESULTS_DIR / f'cv_cnn_fold{k}.npz') as f:
        probs_cnn[fold_all == k], pred_hmm[fold_all == k] = f['probs'], f['pred_hmm']
assert (pred_hmm >= 0).all() and np.allclose(probs_cnn.sum(axis=1), 1, atol=1e-4)
np.savez(RESULTS_DIR / 'cv_cnn_predictions.npz', pid=pid_all, start=start_all, y=y_all, fold=fold_all,
         probs_cnn=probs_cnn, pred_CNN=probs_cnn.argmax(axis=1), **{'pred_CNN+HMM': pred_hmm})
pooled = pd.DataFrame(evaluate('CNN', y_all, probs_cnn.argmax(axis=1)) + evaluate('CNN+HMM', y_all, pred_hmm))
print(pooled.pivot(index='model', columns='task', values=['accuracy', 'macro-F1', 'kappa']).round(3).to_string())

        accuracy        macro-F1          kappa       
task     4-class binary  4-class binary 4-class binary
model                                                 
CNN        0.788  0.912    0.718  0.876   0.683  0.751
CNN+HMM    0.859  0.930    0.781  0.901   0.788  0.802
